# ThermoPhase — Notebook 3: Temperature-Dependent Convex Hull

**Goal:** Construct a temperature-dependent convex hull for a binary model system,
demonstrate the effect of including PyXtal-generated phases, and produce the 
paper's Fig. 3 and Fig. 5.

This notebook demonstrates:
- Building a multi-phase binary system with synthetic data
- Constructing the hull at $T = 0$, $600$, and $1200$ K
- Showing how missing phases affect hull distances
- Generating publication figures

---

In [ ]:
import json, os, tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

from thermophasepy.utils.plotting_utils import (
    fig3_hull_evolution, fig5_coverage_impact, generate_all_figures
)
import thermophasepy.config as cfg

print('Plotting utilities loaded.')

## 1. The temperature-dependent hull: analytical model

We construct a model binary $A$-$B$ phase diagram where some phases become
thermodynamically accessible only at elevated temperatures due to their
higher vibrational entropy.

In [ ]:
from thermophasepy.utils.plotting_utils import fig3_hull_evolution

fig = fig3_hull_evolution(temperatures=[0, 600, 1200])
fig.suptitle('Fig. 3 — Convex hull evolution with temperature', 
             fontsize=9, fontweight='bold', y=1.02)
plt.savefig('fig3_hull_evolution.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig3_hull_evolution.pdf')

### Key observations

1. **At $T = 0$**: The starred phases (Li₃X₂\*, Li₂X₃†) are above the hull — predicted metastable.
2. **At $T = 600$ K**: Configurational and vibrational free energies lower some phases 
   disproportionately, shifting hull positions.
3. **At $T = 1200$ K**: The starred phases are now on or near the hull — they become
   thermodynamically accessible. A static $T=0$ screening would wrongly reject them.

## 2. Effect of PyXtal phase coverage

In [ ]:
fig = fig5_coverage_impact()
fig.suptitle('Fig. 5 — Hull distance before/after PyXtal generation', 
             fontsize=9, fontweight='bold', y=1.02)
plt.savefig('fig5_coverage_impact.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig5_coverage_impact.pdf')

## 3. Build a full synthetic multi-phase database and run the hull

In [ ]:
def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def make_phase(base_dir, formula, sg, e0, theta_D=300, s_conf=0.0):
    """Create a synthetic phase directory."""
    sym = os.path.join(base_dir, formula, sg)
    os.makedirs(sym)
    
    # POSCAR (minimal)
    with open(os.path.join(sym, 'POSCAR'), 'w') as f:
        f.write(f'{formula}\n1.0\n4 0 0\n0 4 0\n0 0 4\n{formula.split("_")[0]}\n1\nDirect\n0 0 0\n')
    
    # Relaxation energy
    write_json(os.path.join(sym, 'relaxation', 'energy.json'),
               {'energy_eV_per_atom': e0})
    
    # Phonon DOS
    omega = np.linspace(0.001, 0.12, 300)
    g = omega**2 * np.exp(-omega / (8.617e-5 * theta_D))
    g /= np.trapz(g, omega)
    write_json(os.path.join(sym, 'phonons', 'dos.json'),
               {'energies_eV': omega.tolist(), 'weights': g.tolist()})
    
    # Site occupancies if disordered
    if s_conf > 0:
        # Back-calculate occupancy from entropy: S = -kB * 2*(x ln x + (1-x) ln(1-x))
        # For x = 0.5: S = kB * ln(2) ≈ 0.693 kB
        write_json(os.path.join(sym, 'disorder', 'site_occupancies.json'), {
            'sites': [{'species': {'A': 0.5, 'B': 0.5}}]
        })
    return sym

# Build a ternary Li-Zr-Cl phase space (model numbers)
db = tempfile.mkdtemp()
phases_spec = [
    # formula,         sg,       E0 (eV/at), θ_D (K), S_conf
    ('Li',            'Im-3m',  -1.65,       195,     0.0),
    ('Zr',            'P6-3mmc',-6.62,       290,     0.0),
    ('LiCl',          'Fm-3m',  -4.01,       315,     0.0),
    ('ZrCl2',         'Pnma',   -4.58,       280,     0.0),
    ('ZrCl4',         'P42-n',  -3.92,       220,     0.0),
    ('Li2ZrCl6',      'R-3m',   -3.85,       250,     0.0),   # TARGET
    ('Li3ZrCl6',      'P-31m',  -3.72,       240,     0.05),  # disordered
    ('LiZrCl5',       'Pnma',   -3.68,       260,     0.0),
]

sym_dirs = {}
for formula, sg, e0, theta, s_conf in phases_spec:
    sd = make_phase(db, formula, sg, e0, theta, s_conf)
    sym_dirs[formula] = sd

print(f'Built {len(sym_dirs)} synthetic phases in {db}')

In [ ]:
from thermophasepy.free_energy import build_default_assembler

temperatures = np.array(list(range(0, 1501, 50)), dtype=float)
asm = build_default_assembler(phonon_mode='harmonic', device='cpu',
                               include_electronic=False,
                               include_magnetic=False)

# Compute G(T) for every phase
G_data = {}
for formula, sd in sym_dirs.items():
    res = asm.compute(sd, temperatures)
    if res['G_eV_per_atom'] is not None:
        G_data[formula] = np.array(res['G_eV_per_atom'])

print('G(T) computed for:', list(G_data.keys()))

In [ ]:
# Simple hull: project Li2ZrCl6 onto the hull of all other phases at each T
# (pymatgen PhaseDiagram would be used in real pipeline — shown here analytically)

target = 'Li2ZrCl6'
competing = [f for f in G_data if f != target]

# G of target vs T
G_target = G_data[target]

# Approximate hull distance: G_target minus G_best_linear_combination
# For this demo we use a weighted average of LiCl + ZrCl4 (decomposition reaction)
# Li2ZrCl6 → 2 LiCl + ZrCl4   → G_hull ≈ (2/9)*G(LiCl) + (1/9)*G(ZrCl4)
G_hull_approx = (2/3)*G_data['LiCl'] + (1/3)*G_data['ZrCl4']  
E_above = (G_target - G_hull_approx) * 1000  # meV/atom

# Verdict
T_op = 1200.0
e_at_T = float(np.interp(T_op, temperatures, E_above))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

# Left: G(T) for all phases
for formula, G in G_data.items():
    style = dict(lw=2.0, color='#D55E00') if formula == target else dict(lw=1.0, alpha=0.6)
    ax1.plot(temperatures, (G - G.min()) * 1000, label=formula, **style)
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel('$G - G_{\\min}$ (meV/atom)')
ax1.set_title('Gibbs free energy (relative)')
ax1.legend(frameon=False, fontsize=6, ncol=2)

# Right: hull distance vs T
ax2.fill_between(temperatures, -50, 0,  color='#d5f5e3', alpha=0.7, label='Stable')
ax2.fill_between(temperatures, 0, 100, color='#fef9e7', alpha=0.7, label='Metastable')
ax2.fill_between(temperatures, 100, 300, color='#fadbd8', alpha=0.7, label='Unstable')
ax2.plot(temperatures, E_above, color='#0072B2', lw=2, label='$\\Delta E_{\\rm hull}$ (Li$_2$ZrCl$_6$)')
ax2.axhline(0, color='#27ae60', lw=0.8, ls='--')
ax2.axhline(100, color='#f39c12', lw=0.8, ls='--')
ax2.axvline(T_op, color='gray', lw=0.8, ls=':')
ax2.annotate(f'{T_op:.0f} K\n{e_at_T:+.0f} meV/at',
             xy=(T_op, e_at_T), xytext=(T_op+100, e_at_T+20),
             fontsize=7, arrowprops=dict(arrowstyle='->', color='gray'))
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('$\\Delta E_{\\rm hull}$ (meV/atom)')
ax2.set_title('Hull distance vs temperature')
ax2.set_ylim(-50, 250)
ax2.legend(frameon=False, fontsize=6)

plt.tight_layout()
plt.savefig('hull_demo.pdf', dpi=300, bbox_inches='tight')
plt.show()
print(f'\nLi2ZrCl6 at {T_op:.0f} K: ΔE_hull = {e_at_T:.1f} meV/atom')
print('Verdict:', 'STABLE' if e_at_T < 0.1 else 'METASTABLE' if e_at_T < 100 else 'UNSTABLE')

## 4. Generate all six paper figures

In [ ]:
os.makedirs('paper_figures', exist_ok=True)
paths = generate_all_figures(output_dir='paper_figures')
print('\nAll paper figures generated:')
for name, path in paths.items():
    print(f'  {name}: {path}')

In [ ]:
from IPython.display import Image
# Display each figure inline
for name, path in paths.items():
    png = path.replace('.pdf', '.png')
    if not os.path.exists(png):
        # Re-render as PNG
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
    print(f'\n=== {name} ===')
    # PNG versions are auto-saved at 300 dpi by plotting_utils
    print(f'  Saved at: {path}')

## 5. Limitations and caveats

The model results above illustrate the framework's behaviour but several
important limitations apply to real-data usage:

1. **Hull approximation**: The hull distance depends on which competing phases
   are included. The PyXtal generation explores 230 space groups × all element
   subsets, but cannot guarantee completeness for complex multi-component systems.

2. **Harmonic approximation for competing phases**: In this notebook, competing
   phases use harmonic $F_{\\rm vib}$. For phases with strongly anharmonic phonons
   (e.g., near a structural phase transition), this introduces systematic errors
   of order 10–50 meV/atom.

3. **VACF trajectory length**: The anharmonic VDOS requires $\\gtrsim 20$ ps
   of production trajectory for reliable spectral resolution. Shorter trajectories
   produce artefactual broadening near $\\omega = 0$ that can trigger false
   vibrational instability flags.

4. **Independent-site configurational entropy**: $F_{\\rm conf}$ uses the ideal
   mixing expression. Correlation effects (short-range order) reduce the actual
   entropy and are not captured here.

See Section 6 of the paper for a full discussion of these limitations and
proposed remedies.

---
**Congratulations** — you've completed all three ThermoPhase notebooks!